# Assistente Medico Hospitalar - Demo Gradio
### Mistral-7B-Instruct-v0.2 + QLoRA (LoRA r=64)

Este notebook carrega o modelo fine-tuned e sobe uma interface Gradio com URL publica temporaria (valida por 72h).

> Prerequisito: Execute em uma sessao do Google Colab com GPU (L4 ou T4).

## 1. Instalacao das Dependencias

In [ ]:
%%time
!pip install -q gradio
!pip install -q -U transformers peft bitsandbytes accelerate

import gradio as gr
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline,
)
from peft import PeftModel
import logging, warnings

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

print(f'Gradio  : {gr.__version__}')
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nao disponivel"}')


## 2. Configuracao

In [ ]:
# ==============================================================================
# CONFIGURACOES
# ==============================================================================
BASE_MODEL_ID   = 'mistralai/Mistral-7B-Instruct-v0.2'
ADAPTER_REPO_ID = 'rodrigoaraujorosa/mistral-7b-assistente-hospitalar-v1'

# Parametros de geracao otimizados (Teste 3 - melhor resultado: 4.0/5)
GEN_CONFIG = dict(
    max_new_tokens    = 1024,
    do_sample         = True,
    temperature       = 0.6,
    top_p             = 0.85,
    repetition_penalty = 1.1,
)

print('Configuracoes definidas.')
print(f'  Modelo base : {BASE_MODEL_ID}')
print(f'  Adaptador   : {ADAPTER_REPO_ID}')
print(f'  Parametros  : {GEN_CONFIG}')


## 3. Carregamento do Modelo

In [ ]:
%%time
# ==============================================================================
# QUANTIZACAO 4-BIT
# ==============================================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Carregando modelo base...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

print('Carregando tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print(f'Aplicando adaptador LoRA de: {ADAPTER_REPO_ID}')
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO_ID)
model.eval()

# Corrige o generation_config para evitar conflito de max_length
model.generation_config = GenerationConfig(
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

vram = torch.cuda.memory_allocated() / 1024**3
print(f'\nModelo pronto! VRAM utilizada: {vram:.2f} GB')


## 4. Funcao de Geracao

In [ ]:
# ==============================================================================
# PIPELINE DE GERACAO
# ==============================================================================
text_generator = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device_map='auto',
)

def gerar_resposta(pergunta, max_tokens, temperature, top_p, rep_penalty):
    if not pergunta.strip():
        return 'Por favor, digite uma pergunta.'

    prompt = f'<s>[INST] {pergunta.strip()} [/INST]'

    result = text_generator(
        prompt,
        max_new_tokens=int(max_tokens),
        do_sample=True,
        temperature=float(temperature),
        top_p=float(top_p),
        repetition_penalty=float(rep_penalty),
        pad_token_id=tokenizer.eos_token_id,
    )
    output = result[0]['generated_text']
    return output.split('[/INST]')[-1].strip()

# Teste rapido antes de subir a interface
print('Testando geracao...')
teste = gerar_resposta(
    'Os altos niveis de procalcitonina na fase inicial apos o transplante de figado pediatrico indicam um resultado pos-operatorio ruim?',
    1024, 0.6, 0.85, 1.1
)
print(f'Resposta de teste:\n{teste}')


## 5. Interface Gradio

A celula abaixo sobe a interface e gera uma **URL publica valida por 72 horas**.
Compartilhe a URL para demonstrar o modelo sem precisar de infraestrutura adicional.

In [ ]:
# ==============================================================================
# INTERFACE GRADIO
# ==============================================================================
exemplos = [
    ['Os altos niveis de procalcitonina na fase inicial apos o transplante de figado pediatrico indicam um resultado pos-operatorio ruim?'],
    ['Os baixos niveis sericos de vitamina D estao associados a depressao pos-AVC?'],
    ['A atividade da autotaxina tem alta precisao para diagnosticar colestase intra-hepatica da gravidez?'],
    ['Os lobar microbleeds estao associados a um declinio no funcionamento executivo em adultos mais velhos?'],
    ['A ceratoprote de Boston fornece uma ampla profundidade de foco?'],
]

with gr.Blocks(theme=gr.themes.Soft(), title='Assistente Medico Hospitalar') as demo:

    gr.Markdown(
        '# Assistente Medico Hospitalar\n'
        'Mistral-7B-Instruct-v0.2 fine-tuned com QLoRA em 211.269 exemplos medicos em portugues brasileiro.\n\n'
        '> Este modelo e destinado a fins educacionais e de pesquisa. '
        'Nao deve ser utilizado para diagnostico ou decisao clinica real.'
    )

    with gr.Row():
        with gr.Column(scale=2):
            pergunta = gr.Textbox(
                label='Pergunta Medica',
                placeholder='Digite sua pergunta clinica aqui...',
                lines=4,
            )
            with gr.Row():
                limpar_btn = gr.Button('Limpar', variant='secondary')
                gerar_btn  = gr.Button('Gerar Resposta', variant='primary')

            resposta = gr.Textbox(
                label='Resposta do Modelo',
                lines=8,
                interactive=False,
            )

        with gr.Column(scale=1):
            gr.Markdown('### Parametros de Geracao')
            max_tokens  = gr.Slider(128, 1024, value=1024, step=128, label='max_new_tokens')
            temperature = gr.Slider(0.1, 1.0,  value=0.6,  step=0.05, label='temperature')
            top_p       = gr.Slider(0.1, 1.0,  value=0.85, step=0.05, label='top_p')
            rep_penalty = gr.Slider(1.0, 1.5,  value=1.1,  step=0.05, label='repetition_penalty')

            gr.Markdown(
                '**Configuracao recomendada (Teste 3 - 4.0/5):**\n'
                '- max_new_tokens: 1024\n'
                '- temperature: 0.6\n'
                '- top_p: 0.85\n'
                '- repetition_penalty: 1.1'
            )

    gr.Markdown('### Exemplos de Perguntas')
    gr.Examples(
        examples=exemplos,
        inputs=pergunta,
        label='Clique para carregar uma pergunta de exemplo',
    )

    gr.Markdown(
        '---\n'
        'Modelo: rodrigoaraujorosa/mistral-7b-assistente-hospitalar-v1 | '
        'Tech Challenge - Fase 3 - 8IADT'
    )

    # Eventos
    gerar_btn.click(
        fn=gerar_resposta,
        inputs=[pergunta, max_tokens, temperature, top_p, rep_penalty],
        outputs=resposta,
    )
    limpar_btn.click(
        fn=lambda: ('', ''),
        outputs=[pergunta, resposta],
    )
    pergunta.submit(
        fn=gerar_resposta,
        inputs=[pergunta, max_tokens, temperature, top_p, rep_penalty],
        outputs=resposta,
    )

# Sobe a interface com URL publica valida por 72h
demo.launch(
    share=True,
    show_error=True,
    quiet=False,
)
